# BÀI TẬP: TITANIC
**Nguồn:** kaggle.com/c/titanic (891 dòng)


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


## A.2. Missing values & Duplicate data

In [ ]:
df.isnull().sum()
df.duplicated().sum()

## A.3. Invalid values

In [ ]:
df.isna().sum()

## A.4. Create a new column
Tạo cột `family_size` = sibsp + parch + 1.

In [17]:
df['family_size'] = df['sibsp'] + df['parch'] + 1


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [ ]:
print(df.mean(numeric_only=True))
print(df.median(numeric_only=True))


## Group 2 — Dispersion

In [ ]:
num_df = df.select_dtypes('int', 'float')
range = num_df.max() - num_df.min()
var = num_df.var()
std = num_df.std()
IQR = num_df.quantile(0.75) - num_df.quantile(0.25)

## Group 3 — Location and Shape

In [ ]:
Q1 = num_df.quantile(0.25)
Q2 = num_df.quantile(0.50)
Q3 = num_df.quantile(0.75)
skew = num_df.skew()
kurt = num_df.kurt()

In [18]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,2
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,2
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,1
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,2
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True,1
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True,1
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False,4
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True,1


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Hạng vé nào có tỷ lệ sống sót cao nhất, chênh lệch bao nhiêu so với hạng thấp nhất?

In [44]:
song_sot = df[df['alive'] == 'yes']
song_sot_theo_hang_ve = song_sot.groupby('class')['alive'].count()
so_kh_theo_hang_ve = df['class'].value_counts()
print('Tỷ lệ sống các hạng vé', (song_sot_theo_hang_ve / so_kh_theo_hang_ve) * 100)
ty_le_song_tung_hang = (song_sot_theo_hang_ve / so_kh_theo_hang_ve) * 100
print('Hạng vé tỷ lệ sống cao nhất', ty_le_song_tung_hang.idxmax())
print('Chênh lệch', ty_le_song_tung_hang.max() - ty_le_song_tung_hang.min(), '% so với hạng thấp nhất')



Tỷ lệ sống các hạng vé class
First     62.962963
Second    47.282609
Third     24.236253
dtype: float64
Hạng vé tỷ lệ sống cao nhất First
Chênh lệch 38.726710417138115 % so với hạng thấp nhất


## Câu hỏi 2: Giới tính hay hạng vé ảnh hưởng đến sống sót mạnh hơn?

In [53]:
song_sot_theo_gt = song_sot.groupby('sex')['alive'].count()
so_kh_theo_gt = df['sex'].value_counts()
ty_le_song_theo_gt = (song_sot_theo_gt/so_kh_theo_gt) * 100
print('Chênh lệch tỷ lệ của giới tính', ty_le_song_theo_gt.max() - ty_le_song_theo_gt.min())
print('Chênh lệch tỷ lệ của hạng vé', ty_le_song_tung_hang.max() - ty_le_song_tung_hang.min())
print('=>Giới tính ảnh hưởng đến sống sót mạnh hơn')

Chênh lệch tỷ lệ của giới tính 55.31300709799203
Chênh lệch tỷ lệ của hạng vé 38.726710417138115
=>Giới tính ảnh hưởng đến sống sót mạnh hơn


## Câu hỏi 3: Vé đắt hơn có thực sự sống sót cao hơn không?

In [ ]:
gia_ve_tb = df.groupby('alive')['fare'].mean()
print('TB giá vé của người sống', gia_ve_tb['yes'])
print('TB giá vé của người chết', gia_ve_tb['no'])
print('Giá vé của người sống chênh', gia_ve_tb['yes'] - gia_ve_tb['no'], 'so với của người chết')
print('=>Giá vé đắt hơn có sự sống sót cao hơn')


TB giá vé của người sống 48.39540760233918
TB giá vé của người chết 22.117886885245902
Giá vé của người sống chênh 26.27752071709328 so với của người chết
Giá vé đắt hơn có sự sống sót cao hơn


## Câu hỏi 4: Gia đình đông người có ảnh hưởng đến khả năng sống sót không?

In [70]:
song_sot = df[df['alive'] == 'yes']
song_sot_theo_gd = song_sot.groupby('family_size')['alive'].count()
so_kh_theo_gd = df['family_size'].value_counts()
ty_le_song_tung_gd = ((song_sot_theo_gd / so_kh_theo_gd) * 100).fillna(0)
print('Tỷ lệ sống các gia đình', ty_le_song_tung_gd)
print('=>Gia đình đông người có ảnh hưởng đến khả năng sống sót')





Tỷ lệ sống các gia đình family_size
1     30.353818
2     55.279503
3     57.843137
4     72.413793
5     20.000000
6     13.636364
7     33.333333
8      0.000000
11     0.000000
dtype: float64
=>Gia đình đông người có ảnh hưởng đến khả năng sống sót


## Câu hỏi 5: Cảng lên tàu (embark_town) nào có tỷ lệ sống sót cao nhất?

In [72]:
song_sot = df[df['alive'] == 'yes']
song_sot_theo_ct = song_sot.groupby('embark_town')['alive'].count()
so_kh_theo_ct = df['embark_town'].value_counts()
ty_le_song_tung_ct = ((song_sot_theo_ct / so_kh_theo_ct) * 100).fillna(0)
print('Tỷ lệ sống các cảng tàu', ty_le_song_tung_ct)
print('Cảng tày có tỷ lệ sống cao nhất là', ty_le_song_tung_ct.idxmax(), 'với', ty_le_song_tung_ct.max())






Tỷ lệ sống các cảng tàu embark_town
Cherbourg      55.357143
Queenstown     38.961039
Southampton    33.695652
dtype: float64
Cảng tày có tỷ lệ sống cao nhất là Cherbourg với 55.35714285714286


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể về dữ liệu Titanic.

*(Viết insight của bạn vào đây...)*